# Summarising the ViT paper — three LangChain patterns

Companion to **M16b — Text summarization with LangChain**.

We download a single arXiv PDF and run **three classic summarisation patterns**
on the same content, so we can see how they differ:

1. **Stuff** — concatenate everything, one LLM call.
2. **Map-reduce** — summarise each chunk in parallel, then summarise the summaries.
3. **Refine** — walk the document chunk-by-chunk, refining the running summary.

The corpus is **"An Image Is Worth 16×16 Words"** (Dosovitskiy et al. 2020) — the
ViT paper. Apt: it's a paper about handling images, and we summarise it with
text-only chains. The deck's slide 14 covers what to do when images themselves
need to be summarised.

All chains are built with **LCEL** (the modern `prompt | llm | parser` pipe
syntax). No deprecated `load_summarize_chain` / `MapReduceDocumentsChain` /
`StuffDocumentsChain` — those will be removed in LangChain 1.0.


In [ ]:
%pip install -q -U langchain langchain-openai langchain-community langchain-text-splitters pypdf tiktoken python-dotenv


In [ ]:
import os, time, asyncio
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from langchain_openai             import ChatOpenAI
from langchain_text_splitters     import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts       import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [ ]:
# Knobs
MODEL_NAME = "gpt-4o-mini"     # cheap, plenty of context for stuff
TEMPERATURE = 0.0              # deterministic summaries
CHUNK_SIZE = 4000              # for map-reduce + refine
CHUNK_OVERLAP = 200

PDF_URL  = "https://arxiv.org/pdf/2010.11929"
PDF_PATH = Path("vit_paper.pdf")

llm = ChatOpenAI(model=MODEL_NAME, temperature=TEMPERATURE)
print(f"model: {MODEL_NAME}  |  temperature={TEMPERATURE}")


## 1. Download and load the paper

`PyPDFLoader` reads a local PDF into a `list[Document]` — one document per page,
with page metadata attached. We download the PDF first (most arXiv URLs are
easier as local files than streamed).


In [ ]:
import urllib.request
if not PDF_PATH.exists():
    print(f"downloading {PDF_URL} ...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print(f"PDF: {PDF_PATH}  ({PDF_PATH.stat().st_size/1024:.0f} KB)")


In [ ]:
loader = PyPDFLoader(str(PDF_PATH))
pages  = loader.load()              # one Document per page

print(f"pages: {len(pages)}")
full_text = "\n\n".join(p.page_content for p in pages)
print(f"total chars: {len(full_text):,}")
print(f"approx tokens: {len(full_text) // 4:,}   (rule of thumb: 1 token ≈ 4 chars)")
print()
print("--- first page preview ---")
print(pages[0].page_content[:500], "...")


## 2. Stuff summarisation — one prompt, one call

Concatenate the whole paper and feed it to the model in a single prompt. With
`gpt-4o-mini`'s 128k context window, the ViT paper (~50 k chars ≈ 13 k tokens)
fits easily.

LCEL: `prompt | llm | parser`. Invoked once with the full text.


In [ ]:
STUFF_PROMPT = ChatPromptTemplate.from_template(
    "Write a concise summary of the following paper. "
    "Cover: the problem it tackles, the proposed approach, the key results, "
    "and the main limitations.\n\n{text}"
)

stuff_chain = STUFF_PROMPT | llm | StrOutputParser()

t0 = time.time()
stuff_summary = stuff_chain.invoke({"text": full_text})
stuff_secs = time.time() - t0

print(f"--- STUFF SUMMARY ({stuff_secs:.1f}s, 1 call) ---")
print(stuff_summary)


## 3. Map-reduce summarisation — parallel map, then reduce

Chunk the document, summarise each chunk **in parallel** via `chain.batch()`,
then concatenate the per-chunk summaries and summarise *those* with a second
prompt (the reduce step).


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_text(full_text)
print(f"chunks: {len(chunks)}  (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")


In [ ]:
# MAP — same chain we'd use for stuff, applied per chunk.
MAP_PROMPT = ChatPromptTemplate.from_template(
    "Summarise the following section of a research paper concisely:\n\n{text}"
)
map_chain = MAP_PROMPT | llm | StrOutputParser()

t0 = time.time()
chunk_summaries = map_chain.batch(                # parallel fan-out
    [{"text": c} for c in chunks],
    config={"max_concurrency": 8},
)
map_secs = time.time() - t0
print(f"map stage: {len(chunk_summaries)} chunks in {map_secs:.1f}s (parallel)")


In [ ]:
# REDUCE — concatenate chunk summaries and ask the model to combine them.
REDUCE_PROMPT = ChatPromptTemplate.from_template(
    "Below are summaries of consecutive sections of a paper. "
    "Combine them into a single coherent summary that covers the problem, "
    "the proposed approach, the key results, and the main limitations.\n\n"
    "{summaries}"
)
reduce_chain = REDUCE_PROMPT | llm | StrOutputParser()

t0 = time.time()
mapreduce_summary = reduce_chain.invoke(
    {"summaries": "\n\n---\n\n".join(chunk_summaries)}
)
reduce_secs = time.time() - t0
mapreduce_secs = map_secs + reduce_secs

print(f"--- MAP-REDUCE SUMMARY ({mapreduce_secs:.1f}s, {len(chunks)+1} calls) ---")
print(mapreduce_summary)


## 4. Refine summarisation — sequential walk

Start with a summary of chunk 1. For each subsequent chunk, ask the LLM to
**refine** the running summary using the new chunk. The summary "knows" what
came before, but each call waits for the previous one — no parallelism.


In [ ]:
INIT_PROMPT = ChatPromptTemplate.from_template(
    "Write a concise summary of the following section:\n\n{text}"
)
init_chain = INIT_PROMPT | llm | StrOutputParser()

REFINE_PROMPT = ChatPromptTemplate.from_template(
    "You have produced a running summary of a paper so far:\n\n"
    "{summary_so_far}\n\n"
    "We have now read the next section:\n\n{text}\n\n"
    "Refine the running summary so it incorporates the new section. "
    "Keep it concise."
)
refine_chain = REFINE_PROMPT | llm | StrOutputParser()


In [ ]:
t0 = time.time()
summary_so_far = init_chain.invoke({"text": chunks[0]})
for i, chunk in enumerate(chunks[1:], start=2):
    summary_so_far = refine_chain.invoke({
        "summary_so_far": summary_so_far,
        "text":           chunk,
    })
    print(f"  refined after chunk {i}/{len(chunks)}")
refine_secs = time.time() - t0
refine_summary = summary_so_far

print(f"\n--- REFINE SUMMARY ({refine_secs:.1f}s, {len(chunks)} sequential calls) ---")
print(refine_summary)


## 5. Side-by-side

Same paper, three patterns. Print all three summaries with their wall times
and call counts so you can compare structure, depth, and any drift.


In [ ]:
import textwrap

def show(label, secs, calls, text):
    print("=" * 80)
    print(f"  {label}   |   wall: {secs:.1f}s   |   LLM calls: {calls}")
    print("=" * 80)
    print(textwrap.fill(text, width=92))
    print()

show("STUFF",      stuff_secs,     1,                  stuff_summary)
show("MAP-REDUCE", mapreduce_secs, len(chunks) + 1,    mapreduce_summary)
show("REFINE",     refine_secs,    len(chunks),        refine_summary)


## 6. Bridge — what about the figures?

The ViT paper has figures (attention maps, embeddings, accuracy plots) that
carry real information. The text-only chains above silently **ignored them** —
`PyPDFLoader` only extracts text. To include them:

| Image type | Strategy |
|---|---|
| Decorative / page headers | drop |
| Charts with embedded labels | OCR (e.g. `pytesseract`) on the image, splice as text near the original caption |
| Photos / complex figures | one VLM call (`gpt-4o`, `claude-sonnet-4-6`, `gemini-2.0`) to caption each figure |
| The whole pipeline is image-heavy | skip text chains — pass pages directly to a multimodal LLM |

**Sketch** of the VLM-caption splice:
```python
# pseudocode — not run here
for fig_image, original_caption, position in extract_figures(pdf):
    vlm_caption = vlm.invoke([
        {"type": "image", "source": {"type": "base64", "data": fig_image}},
        {"type": "text",  "text": "Describe this figure in 1-2 sentences."}
    ])
    enriched_text = enriched_text[:position] + \
        f"\n[Figure: {original_caption} — {vlm_caption}]\n" + \
        enriched_text[position:]
# then chunk + summarise as usual
```

Cost: 1 VLM call per figure (cheap — a paper has 5-15 figures), keeps the
human-authored caption, and gives the summariser something to work with.


## 7. Takeaways

- **Stuff first.** If the doc fits in context (and in 2026 most do), stuff is
  optimal — one call, full cross-document awareness, no patterns lost.
- **Map-reduce when the doc doesn't fit.** Parallel by design. Cross-chunk
  patterns are the casualty.
- **Refine when order matters.** Narrative documents (papers with logical
  flow, transcripts in time order) benefit from the forward-only context.
  Sequential, slow, prone to recency bias.
- **Images need pre-processing.** Captions are free if extracted with the
  PDF; figure content needs OCR or VLM captioning. Or skip text chains
  entirely and use a native multimodal LLM.

> The three patterns are not exclusive — production summarisers often mix
> them (e.g. map-reduce for retrieval-side, refine for the final user-facing
> summary). Pick based on the doc's shape, not on what the framework
> defaults to.
